In [1]:
customer_water_quality = {
    "客户A": {
        "COD": {"value": 1200, "unit": "mg/L"},
        "氨氮": {"value": 80, "unit": "mg/L"},
        "pH": {"value": 7.2, "unit": None}
    },
    "客户B": {
        "COD": {"value": 3500, "unit": "mg/L"},
        "氨氮": {"value": 150, "unit": "mg/L"},
        "pH": {"value": 6.5, "unit": None}
    }
}

In [2]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,ToolMessage,HumanMessage
from langchain_core.tools import tool
import os

load_dotenv()
api_key = os.getenv('SILICONFLOW_API_KEY')
llm = ChatOpenAI(
        api_key=api_key,
    base_url="https://api.siliconflow.cn/v1",
    model="deepseek-ai/DeepSeek-V3.2"
)


In [3]:
@tool
def get_customer_water_quality(customer_name:str)->dict:
    '''用于查找客户的水质信息'''
    customer_name = customer_name.strip()
    if not customer_name:
        return{
            'sccess':False,
            'data':None,
            'error':'客户信息不能为空'
        }
    if customer_name in customer_water_quality:
        return{
            'success':True,
            'data':customer_water_quality[customer_name],
            'error':None
        }
    return{
            'success':False,
            'data':None,
            'error':f'未找到用户{customer_name}'
        }

In [4]:
get_customer_water_quality.invoke({"customer_name": "客户A"})

{'success': True,
 'data': {'COD': {'value': 1200, 'unit': 'mg/L'},
  '氨氮': {'value': 80, 'unit': 'mg/L'},
  'pH': {'value': 7.2, 'unit': None}},
 'error': None}

In [5]:
get_customer_water_quality.invoke({"customer_name": " 客户A "})

{'success': True,
 'data': {'COD': {'value': 1200, 'unit': 'mg/L'},
  '氨氮': {'value': 80, 'unit': 'mg/L'},
  'pH': {'value': 7.2, 'unit': None}},
 'error': None}

In [6]:
get_customer_water_quality.invoke({"customer_name": ""})

{'sccess': False, 'data': None, 'error': '客户信息不能为空'}

In [7]:
@tool
def calculate_price(volume:float,price:float)->dict:
    '''根据污水的单价和数量计算总价'''
    if volume <= 0 or price <=0:
        return{
            'success':False,
            'data':None,
            'error':'单价和水量必须大于0'
        }
    return{
        'success':True,
        'data':{
            "volume":volume,
            "price":price,
            "total_price":volume*price
        },
        'error':None
    }

In [8]:
calculate_price.invoke({
    "volume": 40,
    "price": 30
})

{'success': True,
 'data': {'volume': 40.0, 'price': 30.0, 'total_price': 1200.0},
 'error': None}

In [9]:
calculate_price.invoke({
    "volume": 40,
    "price": -5
})

{'success': False, 'data': None, 'error': '单价和水量必须大于0'}

In [10]:
def check_relevance(query:str,context:str,llm)->bool:
    prompts = f'''
    请判断下面提供的企业内部资料是否能够回答用户问题。

用户问题：
{query}

检索资料：
{context}

判断规则：
- 如果资料中存在能够直接支持回答问题的信息，只回答 YES
- 如果资料与问题无关，或不足以回答，只回答 NO
- 不要输出其他内容
    '''
    response = llm.invoke(prompts)
    return response.content.strip().upper() == "YES"

In [11]:
import os 
os.chdir('..')
from main import initial_rag
from src.query_processing import rewrite_query
from src.retrieval import bm25_search,faiss_search
from src.fusion import rrf_fusion
from src.rag import build_context
bm25, vectorstore, documents,llm = initial_rag()
from langchain_core.tools import tool

@tool
def knowledge_search(query:str)->dict:
    """查询企业污水业务制度、SOP、合同流程等内部知识。"""
    if not query:
        return{
            'success':False,
            'data':None,
            'error':'问题不能为空'
        }
    query = query.strip()
    rewritten_query = rewrite_query(query,llm)
    bm25_result = bm25_search(rewritten_query,bm25,documents,k=5)
    faiss_result = faiss_search(rewritten_query,vectorstore,k=5)
    rrf_results = rrf_fusion(bm25_result,faiss_result,top_k=3)
    context = build_context(rrf_results,top_k=3)
    is_relevance = check_relevance(query,context,llm)
    if not is_relevance:
        return{
            'success':False,
            'data':None,
            'error':'未检索到足够相关的企业内部资料'
        }
    return{
        'success':True,
        'data' : context,
        'error': None
    }

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Administrator\AppData\Local\Temp\jieba.cache
Loading model cost 3.073 seconds.
Prefix dict has been built successfully.


索引或hash无效，重新构建FAISS向量存储。
FAISS对象类型：<class 'langchain_community.vectorstores.faiss.FAISS'>


In [12]:
import inspect
print('rewrite',inspect.signature(rewrite_query))
print('bm225search',inspect.signature(bm25_search))
print('faiss',inspect.signature(faiss_search))
print('rrf',inspect.signature(rrf_fusion))
print('build_context',inspect.signature(build_context))

rewrite (query, llm)
bm225search (query, bm25, documents, k=5)
faiss (query, vectorstore, k=5)
rrf (bm25_results, faiss_results, k=60, top_k=None, bm25_weight=0.5, faiss_weight=1.0)
build_context (rrf_results, top_k=3)


In [13]:
knowledge_search.invoke({
    "query": "外委污水合同签订前需要做哪些准备？"
})

{'success': True,
 'data': '来源：F002_C012\n内容：Section: 4  管理事项与工作程序\nSubsection: 4.2  接收流程\nContent: 4  管理事项与工作程序\n4.2  接收流程\n\n4.2.1  前期对接\n发展与市场部负责收集客户信息，由客户填写《燕山威立雅外委污水新增客户信息表及承诺书》（附件6.3），并对客户的营业执照、排污许可证、开户许可证等资质信息进行审核。\n4.2.2  可行性评估\n客户资质审核通过后，发展与市场部填写《外委污水处理业务评估单》（附件6.2）中“外委污水处理相关信息”等内容，由生产部负责对外委污水接纳的工艺处理的可行性进行评估，包括但不限于是否接收、处理方案、接收地点及接收量等内容；安全环保部负责对环保合规性进行评估，给出是否属于危险废物的评估意见。\n4.2.3  合同谈判及签订\n发展与市场部负责与客户进行接洽、谈判，财务部配合完成成本测算，经理办公室对合同法律合规性进行审核。发展与市场部按照内控流程签订《外委污水处理业务服务协议》，填写《北京燕山威立雅水务有限责任公司新增供应商审批单》，向财务部提出新增客户申请。\n4.2.4  接收及合规达标处理\n生产部负责对已完成《外委污水处理业务服务协议》签订工作的污水接收及达标处理进行管控。各生产单位按生产部要求执行外委污水的接收及达标处理，接收过程中若发现水质异常，须第一时间开展应急处置，并即刻向生产部及公司管理层上报。\n4.2.5  费用结算\n由发展与市场部负责发起费用结算申请，财务部负责进行结算。\n\n----------\n\n来源：F010_C013\n内容：Section: 八、业务流程\nSubsection: 8.4内部评估\nContent: 八、业务流程\n8.4内部评估\n\n目的\n\n根据水质情况，客户的现场情况、资质等相关情况，联合公司相关部门判定是否接该水样。\n\n**8.4.1 Input**\n\n1. 客户水质指标。\n2. 非常明确的客户的情况：污水性质、需处理水量等。\n\n**8.4.3 内部评估的一般流程**\n\n1. 和客户公司联系获取营业执照、排污许可证，确定客户公司是可以合法转运污水的。\n2. 填写内部燕山威立雅外委污水处理业务评估单。\n3. 与安全环

In [14]:
knowledge_search.invoke({
    "query": "火星人的工作时间是什么时候？"
})

{'success': False, 'data': None, 'error': '未检索到足够相关的企业内部资料'}

In [21]:
tools = [get_customer_water_quality,calculate_price,knowledge_search]
tool_map = {tool.name:tool
           for tool in tools}
llm_with_tool = llm.bind_tools(tools)
def agent_loop(question:str,max_step=5):
    message = [SystemMessage(content=
                            '''
                            1. 只完成用户明确提出的任务，不主动扩展到用户未要求的业务判断。
2. 如果用户只要求查询数据或进行计算，只返回对应数据或计算结果。
3. 不得因为已有更多制度资料，就主动增加定价判断、接收判断、合规判断、建议或风险提示，除非用户明确要求。
4. 如果需要调用工具才能完成用户任务，可以调用必要工具，但不要调用与当前任务无关的工具。
5. 所有具体事实必须来自工具返回结果。
6. 工具没有提供的信息，不得自行补充、推测或解释。
7. 如果工具返回失败，明确说明无法获取该信息，不要自行猜测原因。
8. 最终回答应以完成用户原始问题为止，不主动延伸新的业务问题。'''),
              HumanMessage(content=question)]
    actual_tool = []
    for step in range(max_step):
        response = llm_with_tool.invoke(message)
        message.append(response)
        if not response.tool_calls:        
            return  {'answer':response.content,
                    'tools':actual_tool}
        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            actual_tool.append(tool_name)
            tool_args = tool_call['args']
            tool_id = tool_call['id']
            tool_result = tool_map[tool_name].invoke(tool_args)
            print("当前轮次:", step)
            print("调用工具:", tool_name)
            print("调用参数:", tool_args)
            print("工具结果:", tool_result)
            message.append(
                ToolMessage(content=str(tool_result),
                           tool_call_id=tool_id)
            )
    return f'超出最大循环次数，此循环直接结束'

In [18]:
test_questions = [
    "40吨污水，每吨30元，一共多少钱？",
    "客户A的水质数据是什么？",
    "客户C的水质数据是什么？",
    "外委污水合同签订前需要做哪些准备？",
    "火星人的工作制度是什么？",
    "客户A有40吨污水，单价30元，请告诉我水质数据和总处理费用。",
    "客户B有20吨污水，单价50元，请告诉我水质和费用。",
    "0吨污水，每吨30元，一共多少钱？"
]
for question in test_questions:
    print("=" * 60)
    print("问题：", question)
    answer = agent_loop(question)
    print("回答：", answer)

问题： 40吨污水，每吨30元，一共多少钱？
当前轮次: 0
调用工具: calculate_price
调用参数: {'volume': 40, 'price': 30}
工具结果: {'success': True, 'data': {'volume': 40.0, 'price': 30.0, 'total_price': 1200.0}, 'error': None}
回答： 40吨污水按每吨30元计算，总价为1200元。
问题： 客户A的水质数据是什么？
当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户A'}
工具结果: {'success': True, 'data': {'COD': {'value': 1200, 'unit': 'mg/L'}, '氨氮': {'value': 80, 'unit': 'mg/L'}, 'pH': {'value': 7.2, 'unit': None}}, 'error': None}
回答： 客户A的水质数据如下：

| 指标 | 数值 | 单位 |
|---|---|---|
| COD | 1200 | mg/L |
| 氨氮 | 80 | mg/L |
| pH | 7.2 | — |
问题： 客户C的水质数据是什么？
当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户C'}
工具结果: {'success': False, 'data': None, 'error': '未找到用户客户C'}
回答： 无法获取客户C的水质信息：查询返回“未找到用户客户C”。
问题： 外委污水合同签订前需要做哪些准备？
当前轮次: 0
调用工具: knowledge_search
调用参数: {'query': '外委污水合同签订前准备工作'}
工具结果: {'success': True, 'data': '来源：F002_C012\n内容：Section: 4  管理事项与工作程序\nSubsection: 4.2  接收流程\nContent: 4  管理事项与工作程序\n4.2  接收流程\n\n4.2.1  前期对接\n发展与市场部负责收集客户

In [22]:
result = agent_loop(
    "客户B有20吨污水，单价50元，请告诉我水质和费用。"
)

print(result)

当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户B'}
工具结果: {'success': True, 'data': {'COD': {'value': 3500, 'unit': 'mg/L'}, '氨氮': {'value': 150, 'unit': 'mg/L'}, 'pH': {'value': 6.5, 'unit': None}}, 'error': None}
当前轮次: 0
调用工具: calculate_price
调用参数: {'volume': 20, 'price': 50}
工具结果: {'success': True, 'data': {'volume': 20.0, 'price': 50.0, 'total_price': 1000.0}, 'error': None}
{'answer': '客户B的水质与费用查询结果如下：\n\n**水质**\n- COD：3500 mg/L\n- 氨氮：150 mg/L\n- pH：6.5\n\n**费用**\n- 水量：20 吨\n- 单价：50 元/吨\n- 总价：1000 元', 'tools': ['get_customer_water_quality', 'calculate_price']}


In [24]:
test_case = {
    "question": "客户B有20吨污水，单价50元，请告诉我水质和费用。",
    "expected_tools": [
        "get_customer_water_quality",
        "calculate_price"
    ]
}

result = agent_loop(test_case["question"])

print("expected:", test_case["expected_tools"])
print("actual:", result["tools"])
print("pass:", set(test_case["expected_tools"]) ==  set(result["tools"]))

当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户B'}
工具结果: {'success': True, 'data': {'COD': {'value': 3500, 'unit': 'mg/L'}, '氨氮': {'value': 150, 'unit': 'mg/L'}, 'pH': {'value': 6.5, 'unit': None}}, 'error': None}
当前轮次: 0
调用工具: calculate_price
调用参数: {'volume': 20, 'price': 50}
工具结果: {'success': True, 'data': {'volume': 20.0, 'price': 50.0, 'total_price': 1000.0}, 'error': None}
expected: ['get_customer_water_quality', 'calculate_price']
actual: ['get_customer_water_quality', 'calculate_price']
pass: True


In [25]:
test_cases = [
    {
        "question": "40吨污水，每吨30元，一共多少钱？",
        "expected_tools": ["calculate_price"]
    },
    {
        "question": "客户A的水质数据是什么？",
        "expected_tools": ["get_customer_water_quality"]
    },
    {
        "question": "客户C的水质数据是什么？",
        "expected_tools": ["get_customer_water_quality"]
    },
    {
        "question": "外委污水合同签订前需要做哪些准备？",
        "expected_tools": ["knowledge_search"]
    },
    {
        "question": "火星人的工作制度是什么？",
        "expected_tools": ["knowledge_search"]
    },
    {
        "question": "客户A有40吨污水，单价30元，请告诉我水质和总处理费用。",
        "expected_tools": [
            "get_customer_water_quality",
            "calculate_price"
        ]
    },
    {
        "question": "客户B有20吨污水，单价50元，请告诉我水质和费用。",
        "expected_tools": [
            "get_customer_water_quality",
            "calculate_price"
        ]
    },
    {
        "question": "0吨污水，每吨30元，一共多少钱？",
        "expected_tools": ["calculate_price"]
    }
]

In [28]:
results = []
for test in test_cases:
    question  = test['question']
    expected_tools = test['expected_tools']
    result = agent_loop(question)
    actual = result['tools']
    if sorted(actual) == sorted(expected_tools):
        results.append(1)
    else:
        results.append(0)
precision = sum(results)/len(results)
print(precision)

当前轮次: 0
调用工具: calculate_price
调用参数: {'volume': 40, 'price': 30}
工具结果: {'success': True, 'data': {'volume': 40.0, 'price': 30.0, 'total_price': 1200.0}, 'error': None}
当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户A'}
工具结果: {'success': True, 'data': {'COD': {'value': 1200, 'unit': 'mg/L'}, '氨氮': {'value': 80, 'unit': 'mg/L'}, 'pH': {'value': 7.2, 'unit': None}}, 'error': None}
当前轮次: 0
调用工具: get_customer_water_quality
调用参数: {'customer_name': '客户C'}
工具结果: {'success': False, 'data': None, 'error': '未找到用户客户C'}
当前轮次: 0
调用工具: knowledge_search
调用参数: {'query': '外委污水合同签订前需要做哪些准备'}
工具结果: {'success': True, 'data': '来源：F002_C012\n内容：Section: 4  管理事项与工作程序\nSubsection: 4.2  接收流程\nContent: 4  管理事项与工作程序\n4.2  接收流程\n\n4.2.1  前期对接\n发展与市场部负责收集客户信息，由客户填写《燕山威立雅外委污水新增客户信息表及承诺书》（附件6.3），并对客户的营业执照、排污许可证、开户许可证等资质信息进行审核。\n4.2.2  可行性评估\n客户资质审核通过后，发展与市场部填写《外委污水处理业务评估单》（附件6.2）中“外委污水处理相关信息”等内容，由生产部负责对外委污水接纳的工艺处理的可行性进行评估，包括但不限于是否接收、处理方案、接收地点及接收量等内容；安全环保部负责对环保合规性进行评估，给出是否属于危险废物的评估意见。\n4.2.3  合同谈判及